# CRIVO — Gerar exemplos de compreensão (professor com licença aberta)

Gera cerca de 1.000 mensagens de chat por tipo de pergunta e alvo, com o Qwen3-32B-AWQ (Apache 2.0).
As saídas ficam no Drive, na pasta `CRIVO/compreensao`, marcadas com modelo, revisão, data e licença.
Nada disso vai para o repositório nem para o chat. Os rótulos são conferidos depois por `validar_saida.py`.

Requisito: GPU A100 ativa (Ambiente de execução → A100).

In [ ]:
# Célula 0 — GPU e Drive
import subprocess
from pathlib import Path
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("ERRO: sem GPU. Ambiente de execução → A100.")
GPU, MEM = r.stdout.strip().split(", ")
print("GPU:", GPU, MEM, "MB", flush=True)
if int(MEM) < 38000:
    raise SystemExit("ERRO: precisa de A100 (40 GB ou mais).")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
SAIDA = Path("/content/drive/MyDrive/CRIVO/compreensao")
SAIDA.mkdir(parents=True, exist_ok=True)
print("saída:", SAIDA, flush=True)

In [ ]:
# Célula 1 — código do repositório (branch do projeto)
import shutil, subprocess
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/optimistic-fermat-k8kggg",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("commit", COMMIT[:10], flush=True)
subprocess.run(["pip", "install", "-q", "vllm"], check=True)
print("vllm instalado", flush=True)

In [ ]:
# Célula 2 — gerar as respostas do professor (pedidos em compreensao_semantica/geracao/)
import os
os.chdir(REPO)
ENTRADA = REPO / "compreensao_semantica" / "geracao" / "entrada_professor.jsonl.gz"
SAIDA_ARQ = SAIDA / "saida_professor.jsonl.gz"
r = subprocess.run(["python", "experimentos/destilacao_20261014/rodar_modelo.py",
                    "--modelo", "Qwen/Qwen3-32B-AWQ", "--papel", "professor",
                    "--entrada", str(ENTRADA), "--saida", str(SAIDA_ARQ)])
if r.returncode != 0:
    raise SystemExit(f"ERRO ao gerar (código {r.returncode}). Mande o print.")
print("saída gerada:", SAIDA_ARQ, flush=True)

In [ ]:
# Célula 3 — validar: descarta repetidas e iguais à régua, compara com as regras, marca o que revisar
r = subprocess.run(["python", "compreensao_semantica/geracao/validar_saida.py",
                    str(SAIDA_ARQ), str(SAIDA / "saida_professor_modelo.json")])
if r.returncode != 0:
    raise SystemExit("ERRO na validação. Mande o print.")
shutil.copy(REPO / "compreensao_semantica" / "geracao" / "exemplos_candidatos.jsonl", SAIDA / "exemplos_candidatos.jsonl")
print("copiado para o Drive:", SAIDA / "exemplos_candidatos.jsonl", flush=True)
print("PRONTO. Mande o print desta célula para o assistente.", flush=True)